# Authentication Headers for API Requests

When working with APIs, you will almost always need to **authenticate** your requests. Without authentication, anyone could access or modify data — which is obviously not desirable.

There are several common methods of authentication:

| Method | How it works |
|---|---|
| **Username & Password** | Credentials are passed directly via the `auth` parameter in `requests` |
| **Basic Auth Token** | A base64-encoded string of `username:password` is sent in the `Authorization` header |
| **Bearer Token** | A token (often from OAuth or an API key) is sent in the `Authorization` header |

More complex authentication flows (such as OAuth 2.0 with token refresh) exist, but are beyond the scope of this module.

In this notebook, we will practice all three methods listed above using the `requests` library.

## 1. Username & Password Authentication

The simplest way to authenticate is by passing a username and password directly. The `requests.get()` method has an `auth` parameter that accepts a **tuple** of `(username, password)`.

Under the hood, `requests` will encode these credentials and send them as a Basic auth header — but you don't need to worry about that; `requests` handles it for you.

```python
import requests

url = "https://example.com/api/data"
response = requests.get(url, auth=("my_username", "my_password"))
print(response.status_code)
print(response.text)
```

- If the credentials are correct, the server will return a **200** status code.
- If the credentials are wrong, you will typically receive a **401 Unauthorized** status code.

### Exercise 1

Send an authenticated GET request to `https://postman-echo.com/basic-auth` using:
- Username: `"postman"`
- Password: `"password"`

Print the status code and the response text.

**Explanation:** We use the `auth` parameter of `requests.get()` to pass the username and password as a tuple. The `requests` library automatically encodes these into a Basic auth header and sends them with the request. A status code of 200 confirms that the credentials were accepted.

In [ ]:
import requests

url = "https://postman-echo.com/basic-auth"

response = requests.get(url, auth=("postman", "password"))
print(response.status_code)
print(response.text)

### Exercise 2

Send the same GET request to `https://postman-echo.com/basic-auth`, but this time use **incorrect** credentials (e.g. username `"wrong_user"` and password `"wrong_pass"`).

Print the status code and verify that you receive a `401` response.

**Explanation:** By providing invalid credentials, the server rejects the request and returns a 401 Unauthorized status code. This is the standard HTTP response for failed authentication. It is important to always check the status code of your response to detect authentication failures.

In [ ]:
import requests

url = "https://postman-echo.com/basic-auth"

response = requests.get(url, auth=("wrong_user", "wrong_pass"))
print(response.status_code)
print(response.text)

## 2. Basic Auth Tokens

When you use `auth=(username, password)` in `requests`, it actually constructs a **Basic auth header** behind the scenes. You can also build this header yourself.

A Basic auth token is created by:
1. Combining the username and password as `"username:password"`
2. Encoding this string to bytes (UTF-8)
3. Base64-encoding those bytes
4. Placing the result in an `Authorization` header with the prefix `Basic`

```python
import base64

credentials = "my_username:my_password"
encoded = base64.b64encode(credentials.encode("utf-8")).decode("utf-8")
header = {"Authorization": f"Basic {encoded}"}

response = requests.get(url, headers=header)
```

This is exactly what `requests` does when you pass `auth=(username, password)`. Understanding this is useful because some APIs give you a pre-encoded token that you need to pass in the header directly.

See the [Python `base64` documentation](https://docs.python.org/3/library/base64.html) for more details.

### Exercise 3

Use the `base64` module to manually encode the credentials `"postman:password"` into a Basic auth token. Then construct the `Authorization` header and send a GET request to `https://postman-echo.com/basic-auth` using the `headers=` parameter.

Print the status code and the JSON response.

**Explanation:** We first combine the username and password into the format `"username:password"`, then encode that string to bytes using UTF-8, and finally base64-encode it. The result is decoded back to a string so it can be placed in the header. The `base64.b64encode()` function returns bytes, so the `.decode("utf-8")` call converts it to a regular string for use in the f-string.

In [ ]:
import requests
import base64

credentials = "postman:password"
encoded = base64.b64encode(credentials.encode("utf-8")).decode("utf-8")
headers = {"Authorization": f"Basic {encoded}"}

response = requests.get("https://postman-echo.com/basic-auth", headers=headers)
print(response.status_code)
print(response.json())

### Exercise 4

Write a function `make_basic_header(username, password)` that:
- Takes a username and password as arguments
- Returns a dictionary containing the `Authorization` header with the properly encoded Basic auth token

Test your function by using it to send a request to `https://postman-echo.com/basic-auth` with username `"postman"` and password `"password"`. Print the status code.

**Explanation:** We encapsulate the base64 encoding logic into a reusable function. This is good practice because you will often need to authenticate multiple requests in a row, and having a helper function avoids repeating the encoding logic. The function returns a dictionary that can be passed directly to the `headers=` parameter of any `requests` method.

In [ ]:
import requests
import base64


def make_basic_header(username, password):
    credentials = f"{username}:{password}"
    encoded = base64.b64encode(credentials.encode("utf-8")).decode("utf-8")
    return {"Authorization": f"Basic {encoded}"}


headers = make_basic_header("postman", "password")
response = requests.get("https://postman-echo.com/basic-auth", headers=headers)
print(response.status_code)

## 3. Bearer Tokens

Bearer tokens are widely used in modern APIs, especially those that use OAuth 2.0 or API keys. Instead of encoding a username and password, you send a **token** that the server recognizes.

The token is placed in the `Authorization` header with the prefix `Bearer`:

```python
import requests

token = "my-secret-token"
headers = {"Authorization": f"Bearer {token}"}

response = requests.get("https://api.example.com/data", headers=headers)
```

Bearer tokens are typically:
- Obtained from an authentication endpoint (e.g. by logging in with OAuth)
- Provided as an API key by the service provider
- Time-limited and need to be refreshed periodically

The key difference from Basic auth is that Bearer tokens do **not** contain your username or password — they are opaque strings that the server validates independently.

### Exercise 5

Send a GET request to `https://httpbin.org/bearer` with a Bearer token header. Use the token `"my-secret-token"`.

The header should look like: `{"Authorization": "Bearer my-secret-token"}`

Print the status code and the response JSON.

**Explanation:** We construct the `Authorization` header with the `Bearer` prefix followed by the token. The `httpbin.org/bearer` endpoint validates that a Bearer token is present in the header and returns a JSON response confirming the token value. Any non-empty token will be accepted by this test endpoint.

In [ ]:
import requests

token = "my-secret-token"
headers = {"Authorization": f"Bearer {token}"}

response = requests.get("https://httpbin.org/bearer", headers=headers)
print(response.status_code)
print(response.json())

### Exercise 6

Write a function `bearer_request(url, token)` that:
- Takes a URL and a bearer token as arguments
- Sends a GET request with the Bearer token in the `Authorization` header
- If the response status code is 401, prints `"Error: Unauthorized (401). Check your token."` and returns `None`
- Otherwise, returns the JSON response

Test your function twice:
1. With `url="https://httpbin.org/bearer"` and `token="my-secret-token"` (should succeed)
2. With `url="https://httpbin.org/bearer"` and `token=""` (empty token — should fail with 401)

**Explanation:** We wrap the Bearer token request logic into a function that also handles the common 401 error case. Checking the status code before calling `.json()` is important because trying to parse the response body of a 401 response as JSON may raise an exception if the server returns plain text or HTML. Returning `None` on failure allows the caller to easily check whether the request succeeded.

In [ ]:
import requests


def bearer_request(url, token):
    headers = {"Authorization": f"Bearer {token}"}
    response = requests.get(url, headers=headers)

    if response.status_code == 401:
        print("Error: Unauthorized (401). Check your token.")
        return None

    return response.json()


# Test 1: valid token
result = bearer_request("https://httpbin.org/bearer", "my-secret-token")
print(result)

# Test 2: empty token (should fail)
result = bearer_request("https://httpbin.org/bearer", "")
print(result)